# Hyperparameter Tuning For The Final Clean Model

This notebook tunes the final clean safe non-proxy model:
- `full_native_grouped`
- native categorical `CatBoost`

To keep it lightweight while we validate the workflow, it uses a debug subset of the raw CSV by default.
Later, we can remove the `nrows` limit to tune on the full dataset.

## Box 1: Imports And Setup

In [1]:
import json
from pathlib import Path

import pandas as pd
from catboost import CatBoostClassifier
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score, make_scorer, precision_score, recall_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate

CANDIDATE_ROOTS = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
REPO_ROOT = next((path for path in CANDIDATE_ROOTS if (path / "Deliverable3").exists()), Path.cwd())
if REPO_ROOT.name == "Deliverable3":
    REPO_ROOT = REPO_ROOT.parent

FINAL_CLEAN_DIR = REPO_ROOT / "Deliverable3" / "Final_Clean_Model"
OUTPUT_DIR = FINAL_CLEAN_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = Path(r"C:\Users\Vido\Desktop\EPL448\accepted_2007_to_2018q4.csv\accepted_2007_to_2018Q4.csv")
DEBUG_NROWS = 100_000
RANDOM_STATE = 42

print("REPO_ROOT:", REPO_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("CSV_PATH:", CSV_PATH)
print("DEBUG_NROWS:", DEBUG_NROWS)


REPO_ROOT: c:\Users\Vido\Desktop\GitHubRepository\EPL448TeamProject
OUTPUT_DIR: c:\Users\Vido\Desktop\GitHubRepository\EPL448TeamProject\Deliverable3\Final_Clean_Model\outputs
CSV_PATH: C:\Users\Vido\Desktop\EPL448\accepted_2007_to_2018q4.csv\accepted_2007_to_2018Q4.csv
DEBUG_NROWS: 100000


## Box 2: Read The Raw CSV

In [2]:
raw_df = pd.read_csv(CSV_PATH, low_memory=False, nrows=DEBUG_NROWS)
print("raw_df shape:", raw_df.shape)
raw_df.head()


raw_df shape: (100000, 151)


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


## Box 3: Build The Final Clean Native CatBoost Frame

This cell reproduces the final clean preprocessing used for the `full_native_grouped` model.

In [3]:
def parse_numeric_from_text(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series.astype(str).str.extract(r"(\d+)", expand=False), errors="coerce")

def map_emp_title_to_group(emp_title: str) -> str:
    if pd.isna(emp_title):
        return "unknown"
    title = str(emp_title).strip().lower()
    if not title or title in {"nan", "none"}:
        return "unknown"

    keyword_groups = [
        ("retired", ["retired"]),
        ("student", ["student", "intern"]),
        ("self_employed_business_owner", ["self employed", "self-employed", "owner", "founder", "co-owner", "entrepreneur", "small business", "business owner"]),
        ("government_public_safety", ["government", "city of", "county", "state of", "police", "officer", "sheriff", "fire", "army", "navy", "air force", "marine", "military", "federal", "usps", "postal"]),
        ("healthcare", ["nurse", "rn", "doctor", "physician", "medical", "hospital", "health", "dental", "dentist", "pharmacy", "therap", "clinical", "caregiver"]),
        ("education", ["teacher", "professor", "school", "educat", "principal", "instructor", "faculty", "counselor"]),
        ("finance_accounting", ["account", "finance", "financial", "bank", "banker", "credit", "loan", "underwriter", "analyst", "controller", "auditor", "bookkeeper"]),
        ("engineering_it", ["engineer", "developer", "programmer", "software", "network", "systems", "architect", "cyber", "database", "data scientist", "it support", "help desk"]),
        ("management_administration", ["manager", "management", "director", "supervisor", "administrator", "admin", "coordinator", "executive", "operations", "project manager"]),
        ("sales_marketing_real_estate", ["sales", "marketing", "account executive", "realtor", "real estate", "broker", "leasing"]),
        ("transportation_logistics", ["driver", "truck", "transport", "logistics", "warehouse", "delivery", "shipping", "forklift", "dispatcher"]),
        ("construction_skilled_trade", ["construction", "electric", "plumb", "carpent", "mechanic", "machinist", "welder", "contractor", "installer", "maintenance", "hvac"]),
        ("manufacturing_production", ["manufacturing", "production", "assembler", "operator", "factory", "plant"]),
        ("service_retail_hospitality", ["retail", "cashier", "server", "waiter", "waitress", "bartender", "cook", "chef", "restaurant", "hotel", "customer service", "store"]),
    ]

    for group_name, keywords in keyword_groups:
        for keyword in keywords:
            if keyword in title:
                return group_name
    return "other"

def map_purpose_to_group(purpose: str) -> str:
    if pd.isna(purpose):
        return "unknown"
    value = str(purpose).strip().lower()
    if not value or value in {"nan", "none"}:
        return "unknown"

    if value in {"credit_card", "debt_consolidation"}:
        return "debt_refinancing"
    if value in {"home_improvement", "house", "moving"}:
        return "housing_related"
    if value in {"car", "major_purchase", "vacation", "wedding"}:
        return "major_or_discretionary_purchase"
    if value == "medical":
        return "medical_emergency"
    if value == "small_business":
        return "business_investment"
    if value == "educational":
        return "education"
    if value == "renewable_energy":
        return "energy_or_special_project"
    if value == "other":
        return "other"
    return value

FUTURE_COLUMNS = [
    "hardship_flag", "hardship_type", "hardship_reason", "hardship_status", "deferral_term", "hardship_amount", "hardship_start_date", "hardship_end_date",
    "payment_plan_start_date", "hardship_length", "hardship_dpd", "hardship_loan_status", "orig_projected_additional_accrued_interest",
    "hardship_payoff_balance_amount", "hardship_last_payment_amount", "disbursement_method", "debt_settlement_flag", "debt_settlement_flag_date",
    "settlement_status", "settlement_date", "settlement_amount", "settlement_percentage", "settlement_term", "collection_recovery_fee",
    "last_pymnt_amnt", "last_pymnt_d", "next_pymnt_d", "out_prncp", "out_prncp_inv", "policy_code", "recoveries", "total_pymnt",
    "total_pymnt_inv", "total_rec_int", "total_rec_late_fee", "total_rec_prncp", "pymnt_plan"
]
JOINT_COLUMNS = [
    "dti_joint", "annual_inc_joint", "verification_status_joint", "revol_bal_joint", "sec_app_fico_range_low", "sec_app_fico_range_high",
    "sec_app_earliest_cr_line", "sec_app_inq_last_6mths", "sec_app_mort_acc", "sec_app_open_acc", "sec_app_revol_util",
    "sec_app_open_act_il", "sec_app_num_rev_accts", "sec_app_chargeoff_within_12_mths", "sec_app_collections_12_mths_ex_med",
    "sec_app_mths_since_last_major_derog"
]
OBSOLETE_COLUMNS = [
    "id", "initial_list_status", "member_id", "bc_open_to_buy", "last_credit_pull_d", "desc", "funded_amnt_inv", "title", "url", "zip_code", "funded_amnt"
]
PROXY_COLUMNS = ["fico_range_low", "fico_range_high", "grade", "sub_grade", "installment", "int_rate"]
RAW_COLUMNS_REPLACED_LATER = ["issue_d", "earliest_cr_line", "emp_title", "purpose", "addr_state"]

MONTHS_SINCE_COLUMNS = [
    "mths_since_last_delinq", "mths_since_last_record", "mths_since_last_major_derog", "mths_since_rcnt_il", "mo_sin_old_il_acct",
    "mo_sin_rcnt_rev_tl_op", "mo_sin_rcnt_tl", "mths_since_recent_bc", "mths_since_recent_bc_dlq", "mths_since_recent_revol_delinq",
    "num_bc_sats", "num_sats", "all_util", "bc_util", "percent_bc_gt_75", "revol_util", "il_util", "tot_cur_bal",
    "mo_sin_old_rev_tl_op", "mths_since_recent_inq", "pct_tl_nvr_dlq", "tot_hi_cred_lim", "total_bc_limit", "num_tl_120dpd_2m"
]
SAFE_TO_FILL_ZERO = [
    "open_acc_6m", "open_rv_12m", "open_rv_24m", "total_bal_il", "max_bal_bc", "inq_fi", "inq_last_12m", "total_cu_tl", "acc_open_past_24mths",
    "mort_acc", "num_accts_ever_120_pd", "num_actv_bc_tl", "num_actv_rev_tl", "num_bc_tl", "num_il_tl", "num_op_rev_tl", "open_act_il",
    "open_il_12m", "open_il_24m", "tot_coll_amt", "num_tl_30dpd", "num_tl_90g_dpd_24m", "total_il_high_credit_limit"
]
FINAL_LIGHT_MISSING_FILL = [
    "annual_inc", "delinq_2yrs", "inq_last_6mths", "open_acc", "pub_rec", "revol_bal", "total_acc", "collections_12_mths_ex_med",
    "acc_now_delinq", "total_rev_hi_lim", "avg_cur_bal", "chargeoff_within_12_mths", "delinq_amnt", "tax_liens", "pub_rec_bankruptcies",
    "tot_cur_bal", "total_bc_limit", "tot_hi_cred_lim"
]
CATBOOST_CATEGORICAL_COLUMNS = ["home_ownership", "verification_status", "emp_length_cat", "occupation_group", "purpose_group"]
FINAL_NON_FEATURE_COLUMNS = ["target", "loan_status", "application_type", "last_fico_range_high", "last_fico_range_low"]

def build_final_clean_native_frame(raw_df: pd.DataFrame):
    model_df = raw_df.copy()
    model_df = model_df[model_df["loan_status"].isin(["Fully Paid", "Charged Off"])].copy()
    model_df["target"] = (model_df["loan_status"] == "Charged Off").astype(int)
    model_df = model_df[model_df["application_type"] == "Individual"].copy()
    model_df = model_df[model_df["home_ownership"].isin(["MORTGAGE", "RENT", "OWN"])].copy()
    model_df = model_df.reset_index(drop=True)
    source_df = model_df.copy()

    columns_to_drop = FUTURE_COLUMNS + JOINT_COLUMNS + OBSOLETE_COLUMNS + PROXY_COLUMNS + RAW_COLUMNS_REPLACED_LATER
    columns_to_drop = [column for column in columns_to_drop if column in model_df.columns]
    model_df = model_df.drop(columns=columns_to_drop).copy()

    engineered_df = model_df.copy()
    issue_dates = pd.to_datetime(source_df["issue_d"], format="%b-%Y", errors="coerce")
    earliest_credit_dates = pd.to_datetime(source_df["earliest_cr_line"], format="%b-%Y", errors="coerce")
    engineered_df["term"] = parse_numeric_from_text(source_df["term"])
    engineered_df["emp_length"] = parse_numeric_from_text(source_df["emp_length"])
    engineered_df["credit_maturity"] = (
        (issue_dates.dt.year - earliest_credit_dates.dt.year) * 12
        + (issue_dates.dt.month - earliest_credit_dates.dt.month)
    )
    engineered_df["emp_length_cat"] = pd.cut(
        engineered_df["emp_length"],
        bins=[0, 2, 4, 6, 8, 10],
        labels=["1", "2", "3", "4", "5"],
    )
    engineered_df["emp_length_cat"] = engineered_df["emp_length_cat"].cat.add_categories("unknown").fillna("unknown")

    grouped_df = engineered_df.copy()
    grouped_df["occupation_group"] = source_df["emp_title"].apply(map_emp_title_to_group).astype(str)
    grouped_df["purpose_group"] = source_df["purpose"].apply(map_purpose_to_group).astype(str)

    final_df = grouped_df.copy()
    for column in CATBOOST_CATEGORICAL_COLUMNS:
        if column in final_df.columns:
            final_df[column] = final_df[column].fillna("unknown").astype(str)

    for column in MONTHS_SINCE_COLUMNS:
        if column in final_df.columns:
            final_df[f"{column}_is_nan"] = final_df[column].isna().astype(int)
            final_df[column] = final_df[column].fillna(0)

    for column in SAFE_TO_FILL_ZERO:
        if column in final_df.columns:
            final_df[column] = final_df[column].fillna(0)

    for column in FINAL_LIGHT_MISSING_FILL:
        if column in final_df.columns:
            final_df[column] = final_df[column].fillna(0)

    X_final = final_df.drop(columns=[column for column in FINAL_NON_FEATURE_COLUMNS if column in final_df.columns]).copy()
    y_final = final_df["target"].astype(int).copy()

    for column in X_final.columns:
        if column not in CATBOOST_CATEGORICAL_COLUMNS:
            X_final[column] = pd.to_numeric(X_final[column], errors="coerce").fillna(0)

    X_native = X_final.drop(columns=["emp_length"], errors="ignore").copy()
    cat_features = [column for column in CATBOOST_CATEGORICAL_COLUMNS if column in X_native.columns]
    return X_native, y_final, cat_features

X_native, y_final, cat_features = build_final_clean_native_frame(raw_df)
print("X_native shape:", X_native.shape)
print("y_final shape:", y_final.shape)
print("cat_features:", cat_features)


X_native shape: (87496, 99)
y_final shape: (87496,)
cat_features: ['home_ownership', 'verification_status', 'emp_length_cat', 'occupation_group', 'purpose_group']


## Box 4: Build A Balanced Tuning Dataset

To stay consistent with the clean experimentation path, we tune on a balanced dataset built from the current working sample.

In [4]:
positive_indices = y_final[y_final == 1].index.tolist()
negative_indices = y_final[y_final == 0].sample(n=len(positive_indices), random_state=RANDOM_STATE).index.tolist()
balanced_indices = pd.Index(positive_indices + negative_indices)

X_balanced = X_native.loc[balanced_indices].reset_index(drop=True)
y_balanced = y_final.loc[balanced_indices].reset_index(drop=True)

shuffle_order = y_balanced.sample(frac=1.0, random_state=RANDOM_STATE).index
X_balanced = X_balanced.loc[shuffle_order].reset_index(drop=True)
y_balanced = y_balanced.loc[shuffle_order].reset_index(drop=True)

print("X_balanced shape:", X_balanced.shape)
display(y_balanced.value_counts().sort_index().rename_axis("target").to_frame("count"))


X_balanced shape: (35012, 99)


,count
target,
0,17506
1,17506


## Box 5: Baseline Cross-Validation

This is the current final clean-model parameter set before tuning.

In [5]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "accuracy": make_scorer(accuracy_score),
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "weighted_f1": make_scorer(f1_score, average="weighted", zero_division=0),
}

baseline_params = {
    "depth": 6,
    "iterations": 500,
    "learning_rate": 0.05,
    "l2_leaf_reg": 3,
}
baseline_model = CatBoostClassifier(
    **baseline_params,
    loss_function="Logloss",
    verbose=0,
    random_state=RANDOM_STATE,
    allow_writing_files=False,
)
baseline_cv = cross_validate(
    baseline_model,
    X_balanced,
    y_balanced,
    cv=cv,
    scoring=scoring,
    params={"cat_features": cat_features},
    n_jobs=1,
)
baseline_summary_df = pd.DataFrame([{
    "model_version": "baseline",
    "accuracy_mean": float(baseline_cv["test_accuracy"].mean()),
    "precision_mean": float(baseline_cv["test_precision"].mean()),
    "recall_mean": float(baseline_cv["test_recall"].mean()),
    "weighted_f1_mean": float(baseline_cv["test_weighted_f1"].mean()),
    "fit_time_mean_sec": float(baseline_cv["fit_time"].mean()),
}])
display(baseline_summary_df)


,model_version,accuracy_mean,precision_mean,recall_mean,weighted_f1_mean,fit_time_mean_sec
0,baseline,0.673283,0.678887,0.657661,0.673201,27.115372


## Box 6: Grid Search

This is a small, focused grid for the final clean model.

In [6]:
param_grid = {
    "depth": [4, 6, 8],
    "iterations": [300, 500],
    "learning_rate": [0.03, 0.05],
    "l2_leaf_reg": [3],
}

grid_model = CatBoostClassifier(
    loss_function="Logloss",
    verbose=0,
    random_state=RANDOM_STATE,
    allow_writing_files=False,
)
grid_search = GridSearchCV(
    estimator=grid_model,
    param_grid=param_grid,
    scoring=scoring,
    refit="weighted_f1",
    cv=cv,
    n_jobs=1,
    verbose=1,
    return_train_score=False,
)
grid_search.fit(X_balanced, y_balanced, cat_features=cat_features)
grid_results_df = pd.DataFrame(grid_search.cv_results_).sort_values(["rank_test_weighted_f1", "mean_test_recall"], ascending=[True, False]).reset_index(drop=True)
grid_results_display_df = grid_results_df[[
    "param_depth", "param_iterations", "param_learning_rate", "param_l2_leaf_reg",
    "mean_test_accuracy", "mean_test_precision", "mean_test_recall", "mean_test_weighted_f1",
    "rank_test_weighted_f1",
]].copy()
best_params = {
    "depth": int(grid_search.best_params_["depth"]),
    "iterations": int(grid_search.best_params_["iterations"]),
    "learning_rate": float(grid_search.best_params_["learning_rate"]),
    "l2_leaf_reg": int(grid_search.best_params_["l2_leaf_reg"]),
}
best_score = float(grid_search.best_score_)

display(grid_results_display_df.head(10))
print("Best params:", best_params)
print("Best weighted F1:", best_score)


Fitting 3 folds for each of 12 candidates, totalling 36 fits


,param_depth,param_iterations,param_learning_rate,param_l2_leaf_reg,mean_test_accuracy,mean_test_precision,mean_test_recall,mean_test_weighted_f1,rank_test_weighted_f1
0,8,500,0.03,3,0.673998,0.679999,0.657375,0.673906,1
1,6,300,0.05,3,0.673769,0.680272,0.655775,0.673662,2
2,6,500,0.05,3,0.673283,0.678887,0.657661,0.673201,3
3,8,300,0.05,3,0.672969,0.679107,0.655833,0.672872,4
4,6,500,0.03,3,0.672798,0.679934,0.652976,0.672669,5
5,8,300,0.03,3,0.672712,0.680012,0.652462,0.672577,6
6,4,500,0.05,3,0.672455,0.678978,0.654290,0.672345,7
7,8,500,0.05,3,0.671998,0.676783,0.658460,0.671937,8
8,4,500,0.03,3,0.671570,0.679531,0.649435,0.671408,9
9,4,300,0.05,3,0.671427,0.679222,0.649778,0.671272,10


Best params: {'depth': 8, 'iterations': 500, 'learning_rate': 0.03, 'l2_leaf_reg': 3}
Best weighted F1: 0.6739061929489183


## Box 7: Evaluate The Best Tuned Model

In [7]:
tuned_model = CatBoostClassifier(
    **best_params,
    loss_function="Logloss",
    verbose=0,
    random_state=RANDOM_STATE,
    allow_writing_files=False,
)
tuned_cv = cross_validate(
    tuned_model,
    X_balanced,
    y_balanced,
    cv=cv,
    scoring=scoring,
    params={"cat_features": cat_features},
    n_jobs=1,
)
tuned_summary_df = pd.DataFrame([{
    "model_version": "tuned",
    "accuracy_mean": float(tuned_cv["test_accuracy"].mean()),
    "precision_mean": float(tuned_cv["test_precision"].mean()),
    "recall_mean": float(tuned_cv["test_recall"].mean()),
    "weighted_f1_mean": float(tuned_cv["test_weighted_f1"].mean()),
    "fit_time_mean_sec": float(tuned_cv["fit_time"].mean()),
}])

comparison_df = pd.concat([baseline_summary_df, tuned_summary_df], ignore_index=True)
comparison_df["weighted_f1_change_vs_baseline"] = comparison_df["weighted_f1_mean"] - baseline_summary_df.loc[0, "weighted_f1_mean"]
display(comparison_df)


,model_version,accuracy_mean,precision_mean,recall_mean,weighted_f1_mean,fit_time_mean_sec,weighted_f1_change_vs_baseline
0,baseline,0.673283,0.678887,0.657661,0.673201,27.115372,0.000000
1,tuned,0.673998,0.679999,0.657375,0.673906,33.606535,0.000705


## Box 8: Save Outputs

In [8]:
summary_output_path = OUTPUT_DIR / "final_clean_catboost_tuning_summary.csv"
grid_output_path = OUTPUT_DIR / "final_clean_catboost_tuning_grid_results.csv"
params_output_path = OUTPUT_DIR / "final_clean_catboost_tuning_best_params.json"

comparison_df.to_csv(summary_output_path, index=False)
grid_results_display_df.to_csv(grid_output_path, index=False)
params_output_path.write_text(json.dumps(best_params, indent=2), encoding="utf-8")

print("Saved summary to:", summary_output_path)
print("Saved grid results to:", grid_output_path)
print("Saved best params to:", params_output_path)


Saved summary to: c:\Users\Vido\Desktop\GitHubRepository\EPL448TeamProject\Deliverable3\Final_Clean_Model\outputs\final_clean_catboost_tuning_summary.csv
Saved grid results to: c:\Users\Vido\Desktop\GitHubRepository\EPL448TeamProject\Deliverable3\Final_Clean_Model\outputs\final_clean_catboost_tuning_grid_results.csv
Saved best params to: c:\Users\Vido\Desktop\GitHubRepository\EPL448TeamProject\Deliverable3\Final_Clean_Model\outputs\final_clean_catboost_tuning_best_params.json
